# Day 040 — Exercise 2: narrate_top_groups

**What you'll build:** `narrate_top_groups(groups_df, group_col, value_col, model) -> str` — take a `top_groups` DataFrame, convert it to a list of dicts with `df.to_dict(orient='records')`, build a prompt, and return a 2-3 sentence narrative about the rankings.

**Why it matters:** A table of product revenue totals requires the reader to do the comparison themselves. A sentence that says 'Gadget dominates at \$1,800 — nearly double the next product' does the analysis for them. The `orient='records'` format gives the LLM a clean, row-by-row JSON structure.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json
import ollama
import pandas as pd
import io


def distribution_summary(df, col):
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count': int(s.count()), 'mean': round(float(s.mean()), 4),
            'std': round(float(s.std()), 4), 'min': float(s.min()),
            'q25': float(s.quantile(0.25)), 'median': float(s.quantile(0.50)),
            'q75': float(s.quantile(0.75)), 'max': float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count': int(s.count()), 'unique': int(s.nunique()),
        'top': str(counts.index[0]) if len(counts) else None,
        'top_freq': int(counts.iloc[0]) if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }


def top_groups(df, group_col, value_col, n=5):
    return (
        df.groupby(group_col)[value_col]
        .agg(total='sum', mean='mean', count='count')
        .reset_index()
        .nlargest(n, 'total')
        .reset_index(drop=True)
    )


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']


import json
import ollama

def summarize_column(col_name: str, stats: dict,
                     model: str = 'llama3.2') -> str:
    prompt = (
        f"You are a concise data analyst. Describe the column '{col_name}' "
        f"in 1-2 clear sentences for a non-technical reader.\n\n"
        f"Statistics:\n{json.dumps(stats, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()

## Your Implementation

In [ ]:
def narrate_top_groups(groups_df, group_col: str, value_col: str,
                       model: str = 'llama3.2') -> str:
    """
    Narrate a top_groups DataFrame as a plain-English ranking summary.

    Args:
        groups_df  — DataFrame from top_groups() with total/mean/count cols
        group_col  — name of the grouping column (e.g. 'product')
        value_col  — name of the value column (e.g. 'revenue')
        model      — Ollama model to use
    Returns:
        str        — 2-3 sentence narrative about the rankings
    """
    # TODO: records = groups_df.to_dict(orient='records')
    # TODO: build a prompt mentioning group_col and value_col,
    #       include json.dumps(records, indent=2)
    # TODO: call ollama.chat and return stripped content
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0
    _result = None

    # Check 1: function defined
    try:
        assert 'narrate_top_groups' in globals()
        passed += 1; print('\u2705 Check 1: narrate_top_groups is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a string (Ollama call)
    try:
        _top = top_groups(SALES_DF, 'product', 'revenue', n=4)
        _result = narrate_top_groups(_top, 'product', 'revenue')
        assert isinstance(_result, str), \
            f'expected str, got {type(_result).__name__}'
        passed += 1; print('\u2705 Check 2: returns a string')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: response is non-empty
    try:
        assert len(_result.strip()) > 0, 'response is empty'
        passed += 1; print('\u2705 Check 3: response is non-empty')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: response is substantial (> 50 chars for 2-3 sentences)
    try:
        _n = len(_result.strip())
        assert _n > 50, \
            f'response too short ({_n} chars) — expected 2-3 sentences'
        passed += 1; print(f'\u2705 Check 4: response is {_n} chars')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: also works for a different grouping (region)
    try:
        _top_r = top_groups(SALES_DF, 'region', 'revenue', n=4)
        _r_result = narrate_top_groups(_top_r, 'region', 'revenue')
        assert isinstance(_r_result, str) and len(_r_result.strip()) > 0
        passed += 1; print('\u2705 Check 5: also works for region grouping')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import json
import ollama

def narrate_top_groups(groups_df, group_col: str, value_col: str,
                       model: str = 'llama3.2') -> str:
    records = groups_df.to_dict(orient='records')
    prompt = (
        f"You are a concise data analyst. Write 2-3 sentences about which "
        f"'{group_col}' groups have the highest '{value_col}' and what stands out.\n\n"
        f"Top groups by {value_col}:\n{json.dumps(records, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()
```

</details>